# IS353 - Lab 3: ego-Facebook, Gn,p và Watts-Strogatz

**MSSV:** 23520588

So sánh mạng Facebook Social Circles ([SNAP ego-Facebook](https://snap.stanford.edu/data/ego-Facebook.html)) với hai mô hình ngẫu nhiên có cùng số đỉnh và số cạnh gần bằng:

- **Gn,p** (Erdős-Rényi): mỗi cặp đỉnh có cạnh với xác suất p, độc lập.
- **Watts-Strogatz (WS):** mỗi đỉnh nối với k hàng xóm trên một vòng, rồi nối lại mỗi cạnh với xác suất p_rewire.

Các đại lượng: phân bố bậc, clustering, độ dài đường đi trung bình, đường kính, thành phần liên thông.


## Cài thư viện


In [ ]:
%pip install networkx matplotlib numpy


## Import


In [ ]:
%matplotlib inline

from pathlib import Path
import gzip
import shutil
import urllib.request

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


## Hàm dùng chung


In [ ]:
def lab_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "BT" / "Lab3",
        Path(r"d:/Study/IS353/BT/Lab3"),
    ]
    for folder in candidates:
        if (folder / "requirement.txt").exists():
            return folder
    raise FileNotFoundError("Không thấy requirement.txt của Lab 3.")


LAB = lab_dir()
FIG = LAB / "figures"
FIG.mkdir(exist_ok=True)


def load_facebook() -> nx.Graph:
    names = [
        LAB / "facebook_combined.txt",
        LAB.parent / "Lab1_2" / "facebook_combined.txt",
        Path.cwd() / "facebook_combined.txt",
        Path.cwd() / "BT" / "Lab1_2" / "facebook_combined.txt",
    ]
    for path in names:
        if path.exists():
            print(f"Đọc {path}")
            return nx.read_edgelist(path, create_using=nx.Graph)

    gz_path = LAB / "facebook_combined.txt.gz"
    txt_path = LAB / "facebook_combined.txt"
    url = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
    print(f"Tải {url}")
    urllib.request.urlretrieve(url, gz_path)
    with gzip.open(gz_path, "rb") as src, open(txt_path, "wb") as dst:
        shutil.copyfileobj(src, dst)
    print(f"Giải nén {txt_path}")
    return nx.read_edgelist(txt_path, create_using=nx.Graph)


def measure(G, name):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    degrees = [d for _, d in G.degree()]
    avg_degree = (2 * m / n) if n else float("nan")
    components = sorted((len(c) for c in nx.connected_components(G)), reverse=True)
    lcc_nodes = max(nx.connected_components(G), key=len)
    H = G.subgraph(lcc_nodes)

    # Một lượt BFS cho cả đường đi trung bình và đường kính.
    print(f"Đang tính đường đi: {name} ({H.number_of_nodes()} đỉnh)...")
    total = 0
    count = 0
    diameter = 0
    for dist in nx.all_pairs_shortest_path_length(H):
        lengths = dist[1]
        for length in lengths.values():
            if length == 0:
                continue
            total += length
            count += 1
            if length > diameter:
                diameter = length
    avg_path = total / count if count else float("nan")
    clustering = nx.average_clustering(G)

    print(f"=== {name} ===")
    print(f"Nodes                          : {n}")
    print(f"Edges                          : {m}")
    print(f"Average degree                 : {avg_degree:.4f}")
    print(f"Average clustering             : {clustering:.6f}")
    print(f"Average path length (LCC)      : {avg_path:.4f}")
    print(f"Diameter (LCC)                 : {diameter}")
    print(f"Connected components           : {len(components)}")
    print(f"Largest component              : {components[0] if components else 0}")
    return {
        "name": name,
        "nodes": n,
        "edges": m,
        "avg_degree": avg_degree,
        "clustering": clustering,
        "avg_path": avg_path,
        "diameter": diameter,
        "components": len(components),
        "lcc": components[0] if components else 0,
        "degrees": degrees,
    }


def degree_prob(degrees):
    values, counts = np.unique(degrees, return_counts=True)
    prob = counts / counts.sum()
    return values, prob


## 1. Compute degree distribution, clustering, path length, diameter, connected components of the ego-Facebook networks.

### Load ego-Facebook

File `facebook_combined.txt` là danh sách cạnh vô hướng, gộp từ 10 ego-network.


In [ ]:
facebook = load_facebook()
print(f"Facebook: {facebook.number_of_nodes()} nodes, {facebook.number_of_edges()} edges")


### Đo ego-Facebook

Phân bố bậc là xác suất P(k) một đỉnh có bậc k. Clustering là trung bình, trên toàn đồ thị, tỉ lệ cặp hàng xóm có cạnh nối nhau. Đường đi trung bình và đường kính tính trên thành phần liên thông lớn nhất. Với mạng này, thành phần đó là cả đồ thị.

Bước đường đi duyệt BFS từ mọi đỉnh, mất vài phút.


In [ ]:
fb = measure(facebook, "ego-Facebook")


## 2. Generate and measure the Gn,p and WS graphs that has similar number of nodes and edges to the ego-Facebook networks.

### Sinh Gn,p và Watts-Strogatz

Gn,p dùng cùng số đỉnh n. Xác suất p = 2m / (n(n-1)) để số cạnh kỳ vọng bằng số cạnh của Facebook. Số cạnh thực tế dao động một chút quanh m.

WS cần k chẵn. k được lấy số chẵn gần bậc trung bình 2m/n nhất, nên số cạnh n*k/2 sát m. Xác suất nối lại p_rewire = 0.1: phần lớn cạnh vẫn là hàng xóm gần, một phần thành đường tắt.


In [ ]:
n = fb["nodes"]
m = fb["edges"]
p_edge = (2 * m) / (n * (n - 1))
avg_k = 2 * m / n
k = int(round(avg_k))
if k % 2 == 1:
    k = k - 1 if (avg_k - (k - 1)) <= ((k + 1) - avg_k) else k + 1
p_rewire = 0.1

print(f"n = {n}, m = {m}")
print(f"Gn,p: p = {p_edge:.6f}")
print(f"WS: k = {k}, p_rewire = {p_rewire}, edges của vòng = {n * k // 2}")

gnp = nx.gnp_random_graph(n, p_edge, seed=42)
ws = nx.watts_strogatz_graph(n, k, p_rewire, seed=42)
print(f"Gn,p thực tế: {gnp.number_of_nodes()} nodes, {gnp.number_of_edges()} edges")
print(f"WS thực tế:   {ws.number_of_nodes()} nodes, {ws.number_of_edges()} edges")


### Đo Gn,p và WS

Cùng các đại lượng như ego-Facebook. Mỗi đồ thị cũng mất vài phút vì đường đi.


In [ ]:
er = measure(gnp, "Gn,p")
ws_stats = measure(ws, "Watts-Strogatz")


## 3. Compare the measurements of the ego-Facebook networks and the corresponding Gn,p and WS graphs

### Present in a table format

### Bảng so sánh


In [ ]:
rows = [
    ("Nodes", "nodes", None),
    ("Edges", "edges", None),
    ("Average degree", "avg_degree", 4),
    ("Average clustering", "clustering", 6),
    ("Average path length (LCC)", "avg_path", 4),
    ("Diameter (LCC)", "diameter", None),
    ("Connected components", "components", None),
    ("Largest component", "lcc", None),
]
header = f"{'Metric':<28}{'Facebook':>14}{'Gn,p':>14}{'WS':>14}"
print(header)
print("-" * len(header))
for label, key, digits in rows:
    values = []
    for stats in (fb, er, ws_stats):
        value = stats[key]
        if digits is None:
            values.append(str(value))
        else:
            values.append(f"{value:.{digits}f}")
    print(f"{label:<28}{values[0]:>14}{values[1]:>14}{values[2]:>14}")


### Plot using linear-linear plot and log-log plot

### Phân bố bậc: linear-linear và log-log

P(k) là tỉ lệ đỉnh có bậc k. Hình linear cho thấy khối đỉnh nằm ở đâu. Hình log-log kéo đuôi ra: nếu đuôi dài gần như một đường dốc, phân bố lệch nặng, không phải một đỉnh quanh bậc trung bình.


In [ ]:
series = [
    ("ego-Facebook", fb["degrees"], "#1F4E79"),
    ("Gn,p", er["degrees"], "#C47B2B"),
    ("Watts-Strogatz", ws_stats["degrees"], "#2E7D4F"),
]

fig, ax = plt.subplots(figsize=(8, 4.5))
for name, degrees, color in series:
    ks, prob = degree_prob(degrees)
    ax.plot(ks, prob, label=name, color=color, linewidth=1.3)
ax.set_xlabel("Degree k")
ax.set_ylabel("P(k)")
ax.set_title("Degree distribution (linear-linear)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "degree_linear.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
for name, degrees, color in series:
    ks, prob = degree_prob(degrees)
    mask = (ks > 0) & (prob > 0)
    ax.loglog(ks[mask], prob[mask], label=name, color=color, linewidth=1.3)
ax.set_xlabel("Degree k")
ax.set_ylabel("P(k)")
ax.set_title("Degree distribution (log-log)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "degree_loglog.png", bbox_inches="tight")
plt.show()
print(f"Saved: {FIG / 'degree_linear.png'}")
print(f"Saved: {FIG / 'degree_loglog.png'}")


### What models best fit the ego-Facebook networks?

ego-Facebook của SNAP có 4039 đỉnh, 88234 cạnh, một thành phần liên thông. Clustering khoảng 0.606, đường đi trung bình khoảng 3.69, đường kính 8. Bậc lệch phải: nhiều đỉnh bậc thấp, vài hub bậc rất cao.

Gn,p có đường đi ngắn và gần như một thành phần lớn, vì bậc trung bình khoảng 44 là đủ để đồ thị liên thông. Clustering của nó xấp xỉ p, cỡ 0.01, thấp hơn Facebook vài chục lần. Trên cả hai hình, P(k) của Gn,p là một đỉnh quanh bậc trung bình rồi rơi nhanh, không có đuôi dài.

Watts-Strogatz giữ các tam giác của vòng hàng xóm, nên clustering vẫn cao sau khi nối lại 10% cạnh. Đường đi cũng ngắn nhờ các cạnh tắt. Đây là kiểu small-world của Facebook: cụm dày và đi từ đỉnh này sang đỉnh kia chỉ vài bước. Bậc của WS vẫn nằm sát k, không tạo hub.

Trong hai mô hình, WS khớp ego-Facebook hơn ở clustering và độ dài đường đi. Gn,p chỉ khớp phần đường đi ngắn và việc mạng gần như liên thông. Cả hai đều không khớp phân bố bậc. Hình linear giấu đuôi Facebook trong một cột thấp. Hình log-log cho thấy đuôi đó kéo dài, trong khi Gn,p và WS đứt sớm.

### What can we conclude from the measurements?

ego-Facebook là mạng small-world có hub. WS giải thích small-world bằng cụm cục bộ cộng một ít đường tắt. Gn,p là mô hình nền cho đồ thị thưa và đường đi ngắn, không giải thích cụm hay hub.
